# Image to Components with a RBF NN

Companion code for the report.
Trains a hybrid Radial-Basis Function Neural Network on a jellybean image, classifying each pixel into one of 5 color components (Background, Orange, Yellow, Green, Purple). The network uses K-Means to find centers, then solves for output weights via least squares regression.

**Required input files** (must be in the same directory as the notebook):
- `jellybeans1.png` — training image
- `jellybeans2.png` — test image


## Data Preparation

In [ ]:
import tensorflow as tf
import cv2
import numpy as np

img = cv2.imread('jellybeans1.png')
img_array = np.array(img)
print(img_array.shape)

w, h, p = img_array.shape
parsed = img_array.reshape(w*h, 3)
parsed.shape


## Generate pixel labels

The dataset has no labels, so a rule-based color classifier assigns each pixel one of 5 classes based on its RGB values.


In [ ]:
def classify_pixel(rgb):
    b, g, r = rgb
    # Background
    if r > 180 and g > 180 and b > 150 and abs(r - g) < 30:
        return 0
    elif 45 < r < 65 and g < 35 and 40 < b < 60:
        return 4
    elif r < 70 and 30 < g < 80 and b > 40:
        return 4
    elif r > 160 and 20 < g < 80 and b < 60:
        return 1
    elif r > 150 and g > 150 and b < 100 and abs(r - g) < 40:
        return 2
    elif g > 100 and r < 100 and b < 100:
        return 3
    else:
        return 0

y = np.array([classify_pixel(pixel) for pixel in parsed])
x_normalized = parsed / 255.0


## Train the RBF NN: K-Means centers + least squares weights

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

num_classes = 5
class_names = ['Background', 'Orange', 'Yellow', 'Green', 'Purple']
num_centers = 50

kmeans = KMeans(init="k-means++", n_clusters=num_centers, n_init=10, random_state=0)
kmeans.fit(x_normalized)
centers = kmeans.cluster_centers_.astype(np.float32)

# Set sigmas to the average distances between center and all other centers
sigmas = np.array([
    np.mean(np.linalg.norm(centers - centers[i], axis=1))
    for i in range(num_centers)
], dtype=np.float32)


## RBF layer

There are no epochs because it is a deterministic calculation when you give static centers to find ideal weights.


In [ ]:
def rbf_layer_batched(X, centers, sigmas, batch_size=1000):
    N, K = len(X), len(centers)
    phi = np.empty((N, K), dtype=np.float32)
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        diff = X[start:end, np.newaxis, :] - centers[np.newaxis, :, :]
        sq_dist = np.sum(diff ** 2, axis=2)
        phi[start:end] = np.exp(-sq_dist / (2.0 * sigmas ** 2))
        del diff, sq_dist
    return phi


## Build phi matrix and solve for weights

In [ ]:
# Build phi matrix
phi_train = rbf_layer_batched(x_normalized.astype(np.float32), centers, sigmas)
phi_train_b = np.hstack([phi_train, np.ones((len(phi_train), 1), dtype=np.float32)])

# Calculate weights with least squares
y_onehot_train = np.eye(num_classes, dtype=np.float32)[y]
solution, _, _, _ = np.linalg.lstsq(phi_train_b, y_onehot_train, rcond=None)
weights = solution[:-1]
bias = solution[-1]

# Delete unused matrices (calculating weights is RAM intensive)
del phi_train_b, y_onehot_train


## Train results: metrics and confusion matrix

In [ ]:
# Model trained. Now starting results
train_pred = np.argmax(phi_train @ weights + bias, axis=1)

# Metrics
train_acc = accuracy_score(y, train_pred)
print(f"Train Accuracy   : {train_acc:.4f} ({train_acc*100:.2f}%)")
print("\n--- Train Classification Report ---")
print(classification_report(y, train_pred, target_names=class_names))

# Confusion matrix with percentages
cm = confusion_matrix(y, train_pred)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_percent, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', values_format='.1f')
plt.title('RBF-NN Confusion Matrix - Train (%)')
plt.tight_layout()
plt.savefig('rbf_confusion_matrix_train.png', dpi=150, bbox_inches='tight')
plt.show()


## Test on a second image

The trained RBF model is applied to `jellybeans2.png` to test generalization.


In [ ]:
img2 = cv2.imread('jellybeans2.png')
img2_array = np.array(img2)
print(img2_array.shape)

w2, h2, p2 = img2_array.shape
parsed2 = img2_array.reshape(w2*h2, 3)
x_normalized2 = parsed2 / 255.0

# Predict using trained RBF model
phi2 = rbf_layer_batched(x_normalized2.astype(np.float32), centers, sigmas)
predictions2 = phi2 @ weights + bias
y_pred2 = np.argmax(predictions2, axis=1)
y_pred2 = y_pred2.reshape(w2, h2)


## Image decomposition: visualize each predicted color class

For the test image, each panel shows the pixels predicted to belong to that class as white, the rest as black.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 14))
axes = axes.flatten()

for i in range(5):
    mask = (y_pred2 == i).astype(np.uint8) * 255
    axes[i].imshow(mask, cmap='gray')
    axes[i].set_title(f'Predicted Class {i}: {class_names[i]}')
    axes[i].axis('off')

axes[5].axis('off')
plt.tight_layout()
plt.savefig('predicted_classification_jellybeans2.png', dpi=150, bbox_inches='tight')
plt.show()
